In [1]:
# 6-17-2026

In [2]:
import xarray as xr
import numpy as np
import pandas as pd
import regionmask

In [3]:
zarr_path = "seasfire_filtered.zarr"

In [4]:
ds_filtered = xr.open_zarr(zarr_path, consolidated=True)

In [5]:
ds_filtered

<xarray.Dataset> Size: 76GB
Dimensions:                         (latitude: 720, longitude: 1440, time: 506)
Coordinates:
  * latitude                        (latitude) float64 6kB 89.88 ... -89.88
  * longitude                       (longitude) float64 12kB -179.9 ... 179.9
  * time                            (time) datetime64[ns] 4kB 2011-01-01 ... ...
Data variables: (12/41)
    area                            (latitude, longitude) float32 4MB dask.array<chunksize=(45, 45), meta=np.ndarray>
    biomes                          (latitude, longitude) float32 4MB dask.array<chunksize=(45, 45), meta=np.ndarray>
    cams_co2fire                    (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    cams_frpfire                    (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    drought_code_max                (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    drought_code_mean               (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    ...                              ...
    t2m_max                         (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    t2m_mean                        (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    t2m_min                         (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    tp                              (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    vpd                             (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    ws10                            (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
Attributes:
    crs:          EPSG:4326
    description:  The SeasFire Cube is a scientific datacube for seasonal fir...
    title:        SeasFire Cube: A Global Dataset for Seasonal Fire Modeling ...

In [6]:
print(list(ds_filtered.data_vars))

['area', 'biomes', 'cams_co2fire', 'cams_frpfire', 'drought_code_max', 'drought_code_mean', 'fcci_ba', 'fcci_ba_valid_mask', 'fcci_fraction_of_burnable_area', 'fcci_fraction_of_observed_area', 'fcci_number_of_patches', 'fwi_max', 'fwi_mean', 'gwis_ba', 'gwis_ba_valid_mask', 'lai', 'lccs_class_1', 'lccs_class_2', 'lccs_class_3', 'lccs_class_4', 'lccs_class_6', 'lccs_class_7', 'lsm', 'lst_day', 'ndvi', 'pop_dens', 'rel_hum', 'skt', 'ssr', 'ssrd', 'sst', 'swvl1', 'swvl2', 'swvl3', 'swvl4', 't2m_max', 't2m_mean', 't2m_min', 'tp', 'vpd', 'ws10']


In [7]:
from rasterio.features import rasterize
import geopandas as gpd
import rioxarray

In [8]:
ecoregions = gpd.read_file("Terrestrial Ecoregions of the World/data/commondata/data0/wwf_terr_ecos.shp")

In [9]:
ecoregions_4326 = ecoregions.to_crs("EPSG:4326")

In [10]:
ecoregions_4326 = ecoregions_4326.dropna(subset=["ECO_ID"])
ecoregions_4326 = ecoregions_4326[ecoregions_4326["ECO_ID"] > 0]

In [11]:
ecoregions_dissolved = ecoregions_4326.dissolve(
    by='ECO_ID', 
    aggfunc={'ECO_NAME': 'first', 'BIOME_1': 'first'}
).reset_index()

In [12]:
ecoregions_dissolved["ECO_ID"] = ecoregions_dissolved["ECO_ID"].astype(int)

In [13]:
ecoregion_mask = regionmask.mask_geopandas(
    ecoregions_dissolved,
    ds_filtered.longitude,
    ds_filtered.latitude,
    numbers="ECO_ID",
    method="rasterize"
)

c:\Users\Yash\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\interactiveshell.py:3699: FutureWarning: The ``method`` argument is internal and  will be removed in the future. Setting the ``method`` (i.e. backend) should not be necessary. Please raise an issue if you require it.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [14]:
mask_values = ecoregion_mask.values
valid_pixels = mask_values[~np.isnan(mask_values)]
ids, counts = np.unique(valid_pixels, return_counts=True)

df_counts = pd.DataFrame({
    'ECO_ID': ids.astype(int),
    'pixel_count': counts
})

In [15]:
metadata = ecoregions_dissolved[['ECO_ID', 'ECO_NAME', 'BIOME_1']]
df_counts = df_counts.merge(metadata, on='ECO_ID', how='left')
df_counts = df_counts.sort_values(by='pixel_count', ascending=False).reset_index(drop=True)

df_counts.head(20)

,ECO_ID,pixel_count,ECO_NAME,BIOME_1
0,80601,10686,East Siberian taiga,None
1,21102,10547,Maudlandia Antarctic desert,None
2,81327,6583,Sahara desert,None
3,21101,6376,Marielandia Antarctic tundra,None
4,80608,6154,Scandinavian and Russian taiga,None
5,80611,4494,West Siberian taiga,None
6,30713,4089,Sahelian Acacia savanna,None
7,81111,4066,Taimyr-Central Siberian tundra,None
8,51115,3660,Middle Arctic tundra,None
9,80605,3542,Northeast Siberian taiga,None


In [16]:
ecoregion_array = ecoregion_mask.fillna(0).astype(np.int32).values

In [17]:
ds_filtered["ecoregion"] = (("latitude", "longitude"), ecoregion_array)

In [18]:
ds_filtered["ecoregion"].attrs = {
    "units": "unitless",
    "long_name": "WWF Terrestrial Ecoregion ID",
    "description": "Spatial mapping of grid cells to the 825 WWF Terrestrial Ecoregions of the World",
    "missing_value": 0
}

In [19]:
print(ds_filtered["ecoregion"])

<xarray.DataArray 'ecoregion' (latitude: 720, longitude: 1440)> Size: 4MB
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(720, 1440), dtype=int32)
Coordinates:
  * latitude   (latitude) float64 6kB 89.88 89.62 89.38 ... -89.38 -89.62 -89.88
  * longitude  (longitude) float64 12kB -179.9 -179.6 -179.4 ... 179.6 179.9
Attributes:
    units:          unitless
    long_name:      WWF Terrestrial Ecoregion ID
    description:    Spatial mapping of grid cells to the 825 WWF Terrestrial ...
    missing_value:  0


In [20]:
zero_count = (ds_filtered["ecoregion"] == 0).sum().item()
zero_count

784077

In [21]:
from dask.diagnostics import ProgressBar

In [22]:
nonzero_count_delayed = (ds_filtered["gwis_ba"] > 0).sum()

In [23]:
with ProgressBar():
    total_nonzero = nonzero_count_delayed.compute()

[########################################] | 100% Completed | 1.48 sms


In [ ]:
total_nonzero # 3590844 out of ~580608000 total cells, under 1% so makes sense

<xarray.DataArray 'gwis_ba' ()> Size: 8B
array(3590844)
Attributes:
    aggregation:      Spatio-Temporal | sum
    creator_notes:    The missing data for the year 2001 has been filled with...
    downloaded_from:  https://gwis.jrc.ec.europa.eu/apps/country.profile/down...
    long_name:        Burned Areas from GWIS
    provider:         Global Wildfire Information System (GWIS)
    units:            hectares (ha)

580608000

In [28]:
import geopandas as gpd
import regionmask
import numpy as np
from shapely.geometry import box

In [29]:
pyrome_gdf = gpd.read_file("pyromes/Pyromes_alldata.shp")

In [30]:
pyrome_gdf['geometry'] = [
    box(lon - 0.25, lat - 0.25, lon + 0.25, lat + 0.25) 
    for lon, lat in zip(pyrome_gdf['long'], pyrome_gdf['lat'])
]

In [31]:
print(pyrome_gdf.columns)

Index(['Number', 'gridID', 'long', 'lat', 'maxBA', 'meanBA', 'medianBA',
       'CVBA', 'meanFRP', 'q95FRP', 'q99FRP', 'q95size', 'fireSeason',
       'wwf_ter', 'log99FRP', 'logBA', 'logCVBA', 'logSize', 'logMaxBA',
       'logFireSea', 'Class5', 'Uncert', 'X1', 'X2', 'geometry'],
      dtype='object')


In [32]:
pyrome_dissolved = pyrome_gdf.dissolve(by='Class5')

pyrome_dissolved = pyrome_dissolved.reset_index()

mask = regionmask.mask_geopandas(
    pyrome_dissolved,
    ds_filtered.longitude,
    ds_filtered.latitude,
    numbers="Class5", 
    method="rasterize"
)

c:\Users\Yash\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\interactiveshell.py:3699: FutureWarning: The ``method`` argument is internal and  will be removed in the future. Setting the ``method`` (i.e. backend) should not be necessary. Please raise an issue if you require it.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [33]:
pyrome_mask = mask.fillna(255).astype(np.uint8)
ds_filtered["pyrome"] = (("latitude", "longitude"), pyrome_mask.values)

In [34]:
ds_filtered["pyrome"].attrs = {
    "units": "unitless",
    "long_name": "pyrome classification ID",
    "description": "Pyrome assignment based on 5 pyromes paper"
}

In [35]:
ds_filtered

<xarray.Dataset> Size: 76GB
Dimensions:                         (latitude: 720, longitude: 1440, time: 506)
Coordinates:
  * latitude                        (latitude) float64 6kB 89.88 ... -89.88
  * longitude                       (longitude) float64 12kB -179.9 ... 179.9
  * time                            (time) datetime64[ns] 4kB 2011-01-01 ... ...
Data variables: (12/43)
    area                            (latitude, longitude) float32 4MB dask.array<chunksize=(45, 45), meta=np.ndarray>
    biomes                          (latitude, longitude) float32 4MB dask.array<chunksize=(45, 45), meta=np.ndarray>
    cams_co2fire                    (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    cams_frpfire                    (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    drought_code_max                (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    drought_code_mean               (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    ...                              ...
    t2m_min                         (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    tp                              (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    vpd                             (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    ws10                            (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    ecoregion                       (latitude, longitude) int32 4MB 0 0 ... 0 0
    pyrome                          (latitude, longitude) uint8 1MB 255 ... 255
Attributes:
    crs:          EPSG:4326
    description:  The SeasFire Cube is a scientific datacube for seasonal fir...
    title:        SeasFire Cube: A Global Dataset for Seasonal Fire Modeling ...

In [36]:
ds_filtered["pyrome"]

<xarray.DataArray 'pyrome' (latitude: 720, longitude: 1440)> Size: 1MB
array([[255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255],
       ...,
       [255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255],
       [255, 255, 255, ..., 255, 255, 255]],
      shape=(720, 1440), dtype=uint8)
Coordinates:
  * latitude   (latitude) float64 6kB 89.88 89.62 89.38 ... -89.38 -89.62 -89.88
  * longitude  (longitude) float64 12kB -179.9 -179.6 -179.4 ... 179.6 179.9
Attributes:
    units:        unitless
    long_name:    pyrome classification ID
    description:  Pyrome assignment based on 5 pyromes paper

In [37]:
pyrome_counts = ds_filtered["pyrome"].stack(points=("latitude", "longitude"))

# unique values and counts
values, counts = np.unique(pyrome_counts, return_counts=True)

for val, count in zip(values, counts):
    print(f"pyrome ID {val}: {count} cells")

pyrome ID 1: 10504 cells
pyrome ID 2: 34500 cells
pyrome ID 3: 25708 cells
pyrome ID 4: 24084 cells
pyrome ID 5: 14412 cells
pyrome ID 255: 927592 cells


In [38]:
ds_chunked = ds_filtered.chunk({"time": -1, "latitude": 45, "longitude": 45})

In [39]:
for var in ds_chunked.variables:
    if "chunks" in ds_chunked[var].encoding:
        del ds_chunked[var].encoding["chunks"]

In [40]:
delayed_export = ds_chunked.to_zarr(
    "seasfire_pyromes_ecoregions.zarr", 
    mode="w", 
    consolidated=True, 
    compute=False
)

In [41]:
with ProgressBar(): # takes ~ 60 min
    delayed_export.compute(scheduler="sync")

[########################################] | 100% Completed | 299.52 s


In [ ]:
# ~~~ below is code for getting fires and outputting as csv, but need to do AFTER agglomerative clustering of ecoregions

In [10]:
import numpy as np
import pandas as pd
from tqdm import tqdm

In [13]:
features_to_exclude = ["fcci_ba_valid_mask", "fcci_fraction_of_observed_area", "fcci_fraction_of_burnable_area", "fcci_number_of_patches"]
features_to_include = [v for v in ds_filtered.data_vars if v not in features_to_exclude]

In [14]:
fire_mask = (ds_filtered["gwis_ba"].notnull()) & (ds_filtered["gwis_ba"] > 0)

In [15]:
time_idx, lat_idx, lon_idx = np.where(fire_mask.values)

In [16]:
data_dict = {
    "time": ds_filtered.time.values[time_idx],
    "latitude": ds_filtered.latitude.values[lat_idx],
    "longitude": ds_filtered.longitude.values[lon_idx],
    "gwis_ba_target": ds_filtered["gwis_ba"].values[time_idx, lat_idx, lon_idx]
}

In [17]:
for var in tqdm(features_to_include):
    dims = ds_filtered[var].dims
    var_data = ds_filtered[var].values
    
    if dims == ("time", "latitude", "longitude"):
        data_dict[var] = var_data[time_idx, lat_idx, lon_idx]
    elif dims == ("latitude", "longitude"):
        data_dict[var] = var_data[lat_idx, lon_idx]
    elif dims == ("time",):
        data_dict[var] = var_data[time_idx]

100%|██████████| 37/37 [02:29<00:00,  4.05s/it]


In [18]:
df = pd.DataFrame(data_dict) # build df

In [ ]:
df.to_csv("fire_events.csv", index=False)

In [1]:
import pandas as pd

In [2]:
df_read = pd.read_csv("fire_events.csv")

In [ ]:
df_read.head()

,time,latitude,longitude,gwis_ba_target,area,biomes,cams_co2fire,cams_frpfire,drought_code_max,drought_code_mean,...,swvl1,swvl2,swvl3,swvl4,t2m_max,t2m_mean,t2m_min,tp,vpd,ws10
0,2011-01-01,43.125,132.875,21.503023,565025340.0,15.0,NaN,0.0,0.000000,0.00000,...,0.337228,0.338602,0.339852,0.377954,263.25803,256.87710,252.17323,0.350181,0.612398,2.974318
1,2011-01-01,41.375,121.625,214.948550,580685900.0,15.0,NaN,0.0,78.867190,78.86719,...,0.329196,0.329674,0.342462,0.293115,268.53876,263.51370,258.96646,0.099479,1.840315,3.200563
2,2011-01-01,41.125,121.375,42.986618,582878300.0,15.0,NaN,0.0,89.208336,89.20833,...,0.263521,0.266909,0.277763,0.233878,269.10895,264.72153,260.44240,0.056045,2.115829,3.845847
3,2011-01-01,38.875,60.375,64.448320,602096640.0,8.0,NaN,0.0,1642.150000,1639.75630,...,0.013903,0.058413,0.131723,0.163531,281.87384,276.91650,272.45175,0.205564,3.421349,3.884015
4,2011-01-01,37.125,61.125,21.473710,616393900.0,8.0,NaN,0.0,1405.640600,1401.58980,...,0.151022,0.108020,0.139074,0.156350,284.79620,278.67374,273.80170,1.654482,3.947975,2.521311
